# csp mutations analysis
This notebook is designed to import and visualise point mutation data aggregated by `nomadic summarize` from the csp gene. The notebook can be run one cell at a time (Shift-Enter) or all together ('Run All' above).

In [ ]:
import sys
import pandas as pd
from pathlib import Path
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import yaml

pd.options.mode.chained_assignment = None

sys.path.append("../functions")
from gene_deletions import DeletionFinder
from workspace import Workspace

# Settings

In [ ]:
# Decide whether you want the outputs to be saved and in which format
save_results = True
save_format = "svg"

# Load workspace
ws = Workspace()

# Pull in categories from the workspace
categories = ws.categories.copy()

# Define where the outputs will be saved
output_dir = Path.cwd() / "results" / ws.name

if save_results:
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"All results will be saved to: {output_dir}")

# Location for reference files for plotting
ref_dir = Path.cwd().parent / "reference"

# Functions

In [ ]:
def calc_binomialprop_se(n, p):
    return np.sqrt(p * (1 - p) / n)

In [ ]:
def annotate_csp_domains(aa_pos: int, 
                         nterm: int = 104,
                         cterm: int = 272) -> str:
    """Annotate CSP domains"""
    if 1 <= aa_pos <= nterm:
        return "nterm"
    elif nterm < aa_pos < cterm:
        return "repeat"
    elif cterm <= aa_pos:
        return "cterm"

# Load data

In [ ]:
raw_seq_data_dir = Path.cwd().parent / "raw_sequencing_data" / "results" / ws.name 
primary = pd.read_csv(raw_seq_data_dir / "table.analysis_set.csv")

variants_df = pd.read_csv(ws.summaries_path / "variants" / "aa_changes.diversity.csv")
merged_df = pd.merge(
    left=primary[["expt_name", "barcode", "n_amp_gr50"]],
    right=variants_df,
    on=["expt_name", "barcode"],
    how="inner",
)

# Summary Prevalence Tables

In [ ]:
# Summarise
csp_summary_df = (
    merged_df
    .query("gene == 'csp'")
    .groupby(["gene", "chrom", "aa_pos", "aa_change"])
    .agg(n_samples=pd.NamedAgg("aa_call", len),
         n_wt=pd.NamedAgg("aa_call", lambda x: sum(x == "wt")),
         n_mixed=pd.NamedAgg("aa_call", lambda x: sum(x == "mixed")),
         n_mut=pd.NamedAgg("aa_call", lambda x: sum(x == "mutant")),
        )
    .sort_values(["aa_change"])
    .reset_index()
)

# Compute prevalences
for key in ["mixed", "mut"]:
    
    # Frac
    csp_summary_df[f"frac_{key}"] = (
        csp_summary_df[f"n_{key}"] / csp_summary_df["n_samples"]
    )
    
    # CI
    csp_summary_df[f"frac_{key}_ci"] = [
        1.96*calc_binomialprop_se(n, p) 
        for n, p in zip(csp_summary_df["n_samples"], csp_summary_df[f"frac_{key}"])
    ]
    
    # Prev.
    csp_summary_df[f"per_{key}"] = 100 * csp_summary_df[f"frac_{key}"]
    csp_summary_df[f"per_{key}_ci"] = 100 * csp_summary_df[f"frac_{key}_ci"]

# Remove zero prevalence
csp_summary_df = csp_summary_df.query("~(per_mut == 0 and per_mixed == 0)")

### Annotate domains

In [ ]:
csp_summary_df["domain"] = [
    annotate_csp_domains(aa_pos)
    for aa_pos in csp_summary_df["aa_pos"]
]

# Remove repeat region mutations
csp_summary_df = csp_summary_df.query("domain != 'repeat'")

# Very few N-terminal, and at low frequency -- just remove
csp_summary_df = csp_summary_df.query("domain != 'nterm'")

### Annotate amino acid property changes

In [ ]:
aa_df = pd.read_csv(ref_dir / "aa_properties.csv")
prop_dict = {
    oc: p 
    for oc, p in zip(aa_df["one_letter_code"], aa_df["side_chain"])
}

In [ ]:
csp_summary_df["from_aa_prop"] = [prop_dict[m[0]] for m in csp_summary_df["aa_change"]]
csp_summary_df["to_aa_prop"] = [prop_dict[m[-1]] for m in csp_summary_df["aa_change"]]

### Grantham Scores

In [ ]:
gran_df = pd.read_csv(ref_dir / "grantham.csv",index_col=0)

csp_summary_df["gran_dist"] = [
    gran_df.loc[m[0]][m[-1]] for m in csp_summary_df["aa_change"]
]

if save_results:
    csp_summary_df.to_csv(f"{output_dir}/table.csp_cterm_summary.csv", index=False)

### Plot

In [ ]:
#Define colour chart
color_dt = {
    "Hydrophobic": "#D3D3D3",  # lightgrey
    "Positive": "#FF0000",  # red
    "Negative": "#0000FF",  # blue
    "Polar Uncharged": "#228B22",  # forestgreen
    "Special": "#DAA520",  # goldenrod
}
# Define size of points
SZ = 80

In [ ]:
n_mutations = csp_summary_df.shape[0]

In [ ]:
fig = plt.figure(figsize=(8, 4))
fig.subplots_adjust(hspace=0.1)

gs = GridSpec(nrows=6, ncols=1)
ax_top = plt.subplot(gs[0])
ax_bottom = plt.subplot(gs[1:], sharex=ax_top)

# Top
ax = ax_top
ax.scatter(
    x=csp_summary_df["aa_change"],
    y=[2] * n_mutations,
    c=[color_dt[p] for p in csp_summary_df["from_aa_prop"]],
    s=SZ,
    ec="black",
    lw=0.5,
)
ax.scatter(
    x=csp_summary_df["aa_change"],
    y=[3] * n_mutations,
    c=[color_dt[p] for p in csp_summary_df["to_aa_prop"]],
    s=SZ,
    ec="black",
    lw=0.5,
)

ax.scatter(
    x=csp_summary_df["aa_change"],
    y=[1] * n_mutations,
    c=csp_summary_df["gran_dist"],
    cmap="Oranges",
    marker="s",
    s=SZ,
    ec="black",
    lw=0.5,
)
ax.set_ylim(0.5, 3.5)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)

legend_elements = [
    Line2D([0], [0], color=color, lw=0, marker="o", label=label)
    for label, color in color_dt.items()
]
ax.legend(
    handles=legend_elements, bbox_to_anchor=(1, 0), loc="upper left", frameon=False
)

# Bottom
ax = ax_bottom
csp_summary_df.plot(
    kind="bar",
    x="aa_change",
    y=["per_mut", "per_mixed"],
    stacked=True,
    width=0.75,
    ec="black",
    lw=0.5,
    color=sns.color_palette("Purples_r", 2),
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("Frequency (%)")

ax.set_axisbelow(True)
ax.yaxis.set_major_locator(plt.MultipleLocator(10))
# ax.yaxis.set_minor_locator(plt.MultipleLocator(10))
ax.grid(ls="dotted", which="minor", alpha=0.5)
ax.grid(ls="dotted", which="major", alpha=0.5)
ax.set_ylim(0, 100)

if save_results:
    fig.savefig(
        f"{output_dir}/plot.csp_cterm_summary.pdf",
        dpi=300,
        pad_inches=0.5,
        bbox_inches="tight",
    )

In [ ]:
fig = plt.figure(figsize=(8, 4))
fig.subplots_adjust(hspace=0.1)

gs = GridSpec(nrows=6, ncols=1)
ax_top = plt.subplot(gs[-1])
ax_bottom = plt.subplot(gs[:-1], sharex=ax_top)

# Top
ax = ax_top
ax.scatter(
    x=csp_summary_df["aa_change"],
    y=[2] * n_mutations,
    c=[color_dt[p] for p in csp_summary_df["from_aa_prop"]],
    s=SZ,
    ec="black",
    lw=0.5,
)
ax.scatter(
    x=csp_summary_df["aa_change"],
    y=[3] * n_mutations,
    c=[color_dt[p] for p in csp_summary_df["to_aa_prop"]],
    s=SZ,
    ec="black",
    lw=0.5,
)

ax.scatter(
    x=csp_summary_df["aa_change"],
    y=[1] * n_mutations,
    c=csp_summary_df["gran_dist"],
    cmap="Oranges",
    marker="s",
    s=SZ,
    ec="black",
    lw=0.5,
)
ax.set_ylim(0.5, 3.5)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.set_xticks(csp_summary_df.index)
ax.set_xticklabels(csp_summary_df["aa_change"], rotation=90)


# Bottom
ax = ax_bottom
csp_summary_df.plot(
    kind="bar",
    x="aa_change",
    y=["per_mut", "per_mixed"],
    stacked=True,
    width=0.75,
    ec="black",
    lw=0.5,
    color=sns.color_palette("Purples_r", 2),
    ax=ax,
)

ax.set_xlabel("")
ax.set_ylabel("No. Samples")

legend_elements = [
    Line2D([0], [0], color=color, lw=0, marker="o", label=label)
    for label, color in color_dt.items()
]
ax.legend(
    handles=legend_elements, bbox_to_anchor=(1, 0), loc="upper left", frameon=False
)

ax.set_axisbelow(True)
ax.grid(ls="dotted", alpha=0.5)

if save_results:
    fig.savefig(
        f"{output_dir}/plot.cterm_mutations.bottom.pdf",
        dpi=300,
        pad_inches=0.5,
        bbox_inches="tight",
    )